# W08A Colab B · 검증·SVD·마지막 앱

딥러닝응용I(추천시스템) · 동덕여자대학교 · 유원상 교수 · 2026년 10월 20일

CPU에서 위부터 실행합니다. A/B는 각각 독립 실행됩니다. GPU·유료 API·필수 파일 업로드가 필요하지 않습니다. **파일 → Drive에 사본 저장**으로 개인 실습본을 만드세요. 설치가 런타임 재시작을 요구하면 재시작 후 첫 셀부터 실행합니다.

실제 5~6주차를 복습합니다. 설명용 수치는 강의노트의 worked example과 같으며 60개 연습문제의 입력·정답과 구별합니다. Python 문제는 **A에 12개, B에 9개**로 나누고 QID를 유지합니다. 전체 실행 성공과 자신의 문제 풀이 완료는 다릅니다.

학습 순서는 관측과 분모 확인 → 예측과 손실 연결 → old 상태의 갱신 → 검증 선택 → 재구성의 의미입니다. 필요한 중간 활동과 앱 기록은 Q06/Q10/Q18/Q26/Q31/Q39/Q48/Q51/Q58에 연결합니다.


## 코드의 입력·상태·출력을 읽는 방법

표의 함수 이름을 외우는 것이 목표가 아닙니다. 아래 정의 링크에서 **입력 조건 → 계산 → 반환값**을 확인합니다. 새 W08A 함수는 기존 함수를 연결하며 학습 알고리즘을 다시 구현하지 않습니다.

| API | 입력 → 출력과 역할 |
|---|---|
| `rating_metrics` | 같은 길이의 실제·예측 1D 배열 → `n`, `mae`, `rmse`; clipping 없음 |
| `sgd_step` | 같은 길이의 old p/q, 한 평점, 평균·편향·α·β → old 예측/잔차/손실과 새 모수; 입력 불변 |
| `MFSGD.fit` | 관측된 user_id/movie_id/rating 표 → 학습된 모델; 미관측은 행에서 제외 |
| `fit_validated` | 겹치지 않는 train/validation 표와 설정 → model/history/best_epoch; 상한까지 학습 후 모든 최적 모수 복원 |
| `predict_rows` | 학습된 모델과 관측 ID 표 → 입력 행 순서의 예측 배열; 모델 변경 없음 |
| `svd_steps` | NaN이 없는 완전한 실수 행렬과 rank → U/s/Vt/Gram/근사/오차; rank 0 허용 |
| `metric_review_demo` | 선택 예측 4개 → 설명 예제의 관측별 표와 지표 |
| `sgd_review_demo` | α·β → 동일 old 상태에서 한 번의 갱신과 비교표 |
| `mf_review_data` | 인자 없음 → 새 합성 train15/validation3 표와 별도 2×2 완전 행렬 |
| `mf_review_experiment` | K·β·epoch 상한·rank → MF 검증·복원 상태와 별도 SVD 결과 dict |
| `mf_review_view` | 설정과 선택 report_fn → 설명, history, 검증예측표, 재구성표, Figure |
| `build_mf_review_app` | 선택 report_fn → 새 Gradio Blocks; 이 단계에서는 서버 시작 없음 |
| `build_svd_app` | 선택 해석 함수 → 이전 W06B의 SVD·MF 누적 앱 |
| `definition_link` | 실제 함수·클래스·메소드와 고정 ref → 그 버전의 GitHub 정의 시작 행 |

`fit`은 상태를 만드는 단계이고 예측은 그 상태를 읽는 단계입니다. validation은 선택에 쓰지만 gradient에는 넣지 않습니다. `fit_validated`는 실제 조기 종료를 하지 않습니다. 사용자 수 M·아이템 수 N·잠재요인 수 K·추천 길이 N_rec·SVD rank r를 구별합니다.


### 실행 환경 준비

고정 버전의 공통 패키지를 가져옵니다. 설치 ref와 아래 정의 링크의 ref는 동일합니다. 로컬 검증에서는 준비된 프로젝트 환경을 사용합니다.


In [ ]:
import os, sys, subprocess, math, base64
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
IN_COLAB = 'google.colab' in sys.modules
COURSE_VERSION = '2026-fall-w08a'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        f'luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.26.0'], check=True)
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
from luna_recsys.mf_sgd import MFSGD, sgd_step
from luna_recsys.mf_evaluation import rating_metrics, fit_validated, predict_rows
from luna_recsys.svd_foundations import svd_steps
from luna_recsys.svd_lab import build_svd_app
from luna_recsys.mf_review import (metric_review_demo, sgd_review_demo, mf_review_data,
    mf_review_experiment, mf_review_view, build_mf_review_app)
from luna_recsys.model_comparison import definition_link
print('Python:', sys.version.split()[0], '| 수업 설치 버전:', COURSE_VERSION)


### 정의 위치 확인

각 링크는 실제 import한 함수·클래스·메소드의 선언 시작 행을 가리킵니다. 입력 규약은 `help(함수)`로도 확인할 수 있습니다.


In [ ]:
apis = [rating_metrics, sgd_step, MFSGD, MFSGD.fit, fit_validated,
        predict_rows, svd_steps, metric_review_demo, sgd_review_demo, mf_review_data,
        mf_review_experiment, mf_review_view, build_mf_review_app, build_svd_app,
        definition_link]
display(Markdown('\n'.join(
    f'- [{fn.__qualname__}]({definition_link(fn, COURSE_VERSION)})' for fn in apis)))


## 계산 전에 보는 관계도

아래 그림은 이 수업을 위해 만든 설명용 도식입니다. QID 연결에 따라 기호·차원·분모를 읽습니다. 그림 바이트를 notebook에 포함하여 Colab의 상대경로에 의존하지 않습니다.


**설명 그림: 06-split-roles** · 강의노트의 같은 그림과 대조합니다.


In [ ]:
display(Image(data=base64.b64decode('...'), width=900))


**설명 그림: 07-validation-checkpoint** · 강의노트의 같은 그림과 대조합니다.


In [ ]:
display(Image(data=base64.b64decode('...'), width=900))


**설명 그림: 09-rank-error** · 강의노트의 같은 그림과 대조합니다.


In [ ]:
display(Image(data=base64.b64decode('...'), width=900))


**설명 그림: 10-reconstruction-vs-prediction** · 강의노트의 같은 그림과 대조합니다.


In [ ]:
display(Image(data=base64.b64decode('...'), width=900))


## 1. 검증으로 선택하고 모든 모수 복원 · Q39 연결

다음 셀은 train15건에서만 평균과 SGD를 계산합니다. validation3건은 epoch 선택에만 사용합니다. `epochs=12`이면 **초기 epoch0부터 12까지 13행**의 기록이 남습니다. 최저 validation RMSE가 처음 나타난 epoch의 P, Q, b, c를 모두 복원합니다. 학습을 중간에 멈추는 early stopping과 구별합니다. W05A의 train+validation 재학습 절차도 가능한 다른 프로토콜이지만, 이번 셀은 W06A의 최적 상태 복원 절차입니다.


In [ ]:
review = mf_review_experiment(K=2, beta=.1, epochs=12, rank=1)
display(review['history'])
display(review['validation_predictions'])
print('선택 epoch:', review['best_epoch'])
print('복원 상태 validation 지표:', review['validation_metrics'])
print('P / Q shape:', review['model'].P_.shape, review['model'].Q_.shape)


## 2. 별도 완전 행렬의 SVD · Q48/Q51 연결

설명 행렬 A=[[4,2],[2,4]]는 빈 칸이 없는 독립 합성 예제입니다. 위 MF 관측표를 0으로 채운 행렬이 아닙니다. Gram 행렬 AᵀA의 고유값은 36과 4이므로 특이값은 6과 2입니다. rank1 근사는 모든 칸이 3, 버린 제곱합은 4, Frobenius 오차는 2입니다.

$$
A=U\Sigma V^\top,\qquad A_r=\sum_{j=1}^r\sigma_j u_jv_j^\top,\qquad
\|A-A_r\|_F^2=\sum_{j>r}\sigma_j^2.
$$

U와 V의 대응 열 부호가 함께 반전되어도 같은 A를 만듭니다. singular value가 0이면 Av/σ를 계산하지 않습니다. 축소 SVD 자체는 모든 특이값을 유지할 수 있으므로 항상 근사인 것은 아닙니다. 잠재축을 자동으로 장르 이름으로 해석하지 않습니다.


In [ ]:
svd_demo = review['svd']
for name in ['A', 'gram', 'U', 's', 'Vt', 'reconstruction']:
    print(name)
    display(pd.DataFrame(np.atleast_2d(svd_demo[name])))
print('재구성오차 / 버린 제곱합 / 보존 비율:', svd_demo['error'], svd_demo['discarded_energy'], svd_demo['energy_fraction'])


## 3. 두 결과의 주장 범위 · Q58 연결

MF의 validation RMSE는 보류한 평점 예측 오차입니다. SVD의 90%는 별도 입력 행렬의 제곱합 보존 비율입니다. 두 실험은 데이터·분모·목적이 달라 숫자만으로 우열을 결론 내리지 않습니다. rank를 바꾸어도 MF의 train·validation 곡선은 변하지 않아야 합니다.


In [ ]:
full_rank = mf_review_experiment(K=2, beta=.1, epochs=12, rank=2)
comparison = pd.DataFrame([{'experiment':'MF validation ratings', 'quantity':'RMSE', 'value':review['validation_metrics']['rmse']}, {'experiment':'separate complete matrix', 'quantity':'Frobenius reconstruction error', 'value':review['svd']['error']}])
display(comparison)
print('rank2 재구성오차:', full_rank['svd']['error'])


## 강의자료와 같은 번호의 Python 연습

문제 코드는 Markdown 안에 두어 의도한 오류나 빈칸이 전체 실행을 중단시키지 않게 했습니다. 문제 바로 아래 실행 셀에 자신의 코드를 씁니다. 안전한 starter의 실행 성공은 문제를 풀었다는 뜻이 아닙니다. 문제 원문·수치·소질문은 강의자료와 동일합니다.


### Q42 · 첫 최솟값을 선택하는 코드

다음 코드는 epoch 0을 포함하고 validation 동점에서 먼저 나온 epoch를 유지하려 한다. (가) 빈칸에 비교 연산자 하나를 넣으라. (나) 출력을 쓰고, 동점에서 왜 다른 epoch로 바뀌지 않는지 설명하라.

```python
val = [1.10, 0.97, 0.89, 0.89, 0.95]
best = 0
for epoch in range(1, len(val)):
    if val[epoch] ____ val[best]:
        best = epoch
print(best, val[best])
```


In [ ]:
# Q42 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q42_my_answer = None


### Q43 · 저장했다고 생각한 최적 상태

아래의 리스트는 선택 epoch의 P·Q·사용자 편향 b·아이템 편향 c를 각각 한 값으로 축약한 것이다. 이후 학습을 흉내 내어 값을 바꾼다. `values[:]`는 리스트를 복사한다. (가) 원 코드의 출력과 오류 원인을 설명하라. (나) best를 저장하는 한 줄을 수정하여 네 상태를 모두 보존하고 수정 후 출력을 쓰라.

```python
state = {'P': [0.4], 'Q': [0.5], 'b': [0.1], 'c': [-0.2]}
best = state
state['P'][0] = 0.9
state['Q'][0] = 0.8
state['b'][0] = 0.3
state['c'][0] = 0.2
restored = best
print(restored)
```


In [ ]:
# Q43 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q43_my_answer = None


### Q44 · 자료 역할로 평균을 계산하기

rows는 (집합 이름, 평점) 목록이다. (가) 학습 평균을 계산하도록 빈칸을 채우라. (나) 출력과 분모를 쓰라. (다) test의 1을 5로 바꾸어도 학습 평균이 같아야 하는 이유를 설명하라.

```python
rows = [('train', 4), ('validation', 5), ('train', 2),
        ('test', 1), ('train', 3)]
chosen = [rating for split, rating in rows if split == ____]
mean = sum(chosen) / ____
print(chosen, mean)
```


In [ ]:
# Q44 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q44_my_answer = None


### Q52 · 첫 SVD 항을 조립하는 코드

아래에서 `*`는 원소별 곱, `@`는 행렬곱이다. `U[:, :rank] * s[:rank]`는 선택한 각 U 열에 해당 특이값을 곱한다. (가) 빈칸에 행렬곱 연산자와 남길 Vt의 행 수를 채우라. (나) 출력과 결과의 크기를 쓰라.

```python
import numpy as np
U = np.array([[1., 0.], [0., 1.]])
s = np.array([8., 3.])
Vt = np.array([[1., 0.], [0., 1.]])
rank = 1
rebuilt = (U[:, :rank] * s[:rank]) ____ Vt[:____, :]
print(rebuilt.tolist())
```


In [ ]:
# Q52 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q52_my_answer = None


### Q53 · 실행되지만 재구성하지 않는 코드

주어진 SVD 인수의 원본은 $A=\begin{bmatrix}0&6\\2&0\end{bmatrix}$이다. `*`는 원소별 곱, `@`는 행렬곱이며 `(U * s)`는 각 열에 특이값을 곱한다. (가) 원 코드의 출력을 쓰고 틀린 이유를 설명하라. (나) rebuilt 한 줄을 수정하여 원본을 재구성하라.

```python
import numpy as np
U = np.array([[1., 0.], [0., 1.]])
s = np.array([6., 2.])
Vt = np.array([[0., 1.], [1., 0.]])
rebuilt = U * s * Vt
print(rebuilt.tolist())
```


In [ ]:
# Q53 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q53_my_answer = None


### Q54 · 출력 크기와 남은 오차 추적

다음 함수의 설명이 주어졌다. `svd(..., full_matrices=False)`는 축소 SVD를 반환하고 s는 특이값의 1차원 배열이다. `diag([a,b])`는 대각행렬, `norm`은 여기서 모든 원소 제곱합의 제곱근이다. (가) 두 print의 출력을 쓰라. (나) rank를 2로 바꾸면 B의 크기와 오차가 어떻게 되는지 설명하라.

```python
import numpy as np
A = np.diag([9., 4.])
U, s, Vt = np.linalg.svd(A, full_matrices=False)
rank = 1
B = (U[:, :rank] * s[:rank]) @ Vt[:rank, :]
print(U.shape, s.shape, Vt.shape, B.shape)
print(round(float(np.linalg.norm(A - B)), 4))
```


In [ ]:
# Q54 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q54_my_answer = None


### Q58 · 앱의 설명 함수를 직접 고치기

SVD 앱은 `build_svd_app(report_fn=my_report)`로 연결한 함수를 계산 뒤 호출한다. d의 `error`는 입력 행렬에 대한 Frobenius 오차, `energy_fraction`은 제곱합 보존 비율이다. 영행렬에서는 비율이 None이다. (가) 두 빈칸에 제공된 키를 넣으라. (나) example의 출력을 쓰라. (다) 앱에서 rank만 바꾸어 실행하고, 선택한 행렬 이름·변경 전후 rank·각 실행의 실제 재구성오차와 보존 비율·내가 수정한 해석 문장을 기록하라. 이 해석이 필요한 이유도 설명하라. 함수 이름은 아래에서 제공되므로 외울 필요가 없다.

```python
def my_report(d):
    error = d[____]
    fraction = d[____]
    ratio = '정의하지 않음' if fraction is None else f'{100*fraction:.1f}%'
    return f'입력 재구성오차={error:.1f}; 제곱합 보존={ratio}. 추천 정확도는 별도 평가가 필요합니다.'

example = {'error': 5.0, 'energy_fraction': 144/169}
print(my_report(example))
# 실습 앱에서 연결: app = build_svd_app(report_fn=my_report)
```


In [ ]:
# Q58 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q58_my_answer = None


### Q59 · 숨긴 정답 대신 대입값을 채점한 코드

truth는 평가를 위해 별도 보관한 설명용 정답이며 학습에는 전달하지 않았다. filled는 한 칸에 0을 대입한 입력이고 hidden은 그 한 칸만 True인 평가 마스크다. `a[mask]`는 True 위치의 값만 선택한다. (가) 원 코드 출력과 잘못된 비교 대상을 지적하라. (나) score 한 줄을 고쳐 실제 숨긴 평점의 RMSE와 출력을 구하라.

```python
import numpy as np
truth = np.array([[3., 4.], [2., 5.]])
filled = np.array([[3., 0.], [2., 5.]])
rebuilt = filled.copy()  # 모든 항을 유지한 정확한 입력 재구성
hidden = np.array([[False, True], [False, False]])
score = np.sqrt(np.mean((filled[hidden] - rebuilt[hidden])**2))
print(f'숨긴 평점 RMSE={score:.1f}')
```


In [ ]:
# Q59 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q59_my_answer = None


### Q60 · 성능이 좋아진 것처럼 보이는 평균 기준선

다음 코드는 모든 test 평점을 하나의 평균으로 예측한다. test는 최종 평가용이며 모형 선택은 끝났다. (가) 원 코드의 평균·test RMSE를 구하고 누수 위치를 찾으라. (나) 평균 계산 한 줄을 고쳐 올바른 두 출력을 쓰라. (다) 수정 후 오차가 커졌다는 이유로 원 코드를 더 좋은 모형이라고 주장할 수 있는지 설명하라.

```python
train = [2.0, 4.0]
test = [5.0, 5.0]
mean = sum(train + test) / len(train + test)
rmse = (sum((y - mean)**2 for y in test) / len(test))**0.5
print(f'mean={mean:.1f}, test RMSE={rmse:.1f}')
```


In [ ]:
# Q60 자신의 풀이 코드: 이 셀을 수정하여 실행하세요.
q60_my_answer = None


## 4. Q58의 원리를 내 앱 문장으로 연결

Q58의 기존 `build_svd_app(report_fn=my_report)`는 SVD 결과 dict를 직접 전달합니다. 아래 `my_report(svd_result)`도 같은 규약으로 작성합니다. 새 통합 앱은 MF와 SVD를 함께 반환하므로 연결할 때 `result["svd"]`만 꺼내는 어댑터를 명시합니다. **보고서 함수 → 앱 생성 → 마지막 실행** 셀 순서로 다시 실행해야 변경된 함수가 새 앱 callback에 연결됩니다. 데이터·분모·학습 상태를 바꾸는 코드는 설명 함수에 넣지 않습니다.


In [ ]:
def my_report(svd_result):
    return 'Q58 기록: 별도 입력 행렬의 재구성오차와 보존 비율을 내 문장으로 설명하세요.'
print(my_report(review['svd']))


### UI와 callback의 연결

통합 앱의 버튼은 K·β·epoch·rank를 callback에 전달합니다. callback은 `mf_review_experiment`의 결과를 어댑터에 전달하고, 어댑터는 그중 SVD dict를 `my_report`로 보냅니다. 반환 문장은 MF·SVD 계산표와 함께 표시됩니다. 기존 `build_svd_app(report_fn=my_report)`를 직접 실행할 때는 중첩 결과가 아니므로 어댑터가 필요하지 않습니다. 정의 링크에서 `button.click`과 `report_fn` 연결을 확인한 뒤 앱을 새 함수로 다시 구성합니다.


In [ ]:
if 'app' in globals():
    app.close()
app = build_mf_review_app(report_fn=lambda result: my_report(result['svd']))
print('앱 생성 완료. 실행은 마지막 셀에서 합니다.')


## 점검 퀴즈

강의노트에 있는 동일한 다섯 확인 항목입니다. 새로운 독립 문항을 추가하지 않습니다.

1. 설명 예제의 MAE와 RMSE가 원래 평점 단위로 해석되는 이유는 무엇인가요? (연결 Q06)
2. 설명 예제의 MF 예측에서 사용자·아이템 편향은 각각 어떤 기준 차이를 나타내나요? (연결 Q18)
3. 설명 예제에서 p를 먼저 갱신한 뒤 새 p로 q를 계산하면 명시된 동시 갱신과 같을까요? (연결 Q31)
4. 설명용 검증 곡선에서 선택 epoch의 번호만 저장하면 충분한가요? (연결 Q39)
5. 설명 예제의 SVD 에너지 보존율 90%를 추천 정확도라고 부를 수 있나요? (연결 Q58)


## 실행 검증과 핵심 정리

검증은 관측 경계, 선택 상태, SVD 재구성, 실제 앱 callback 연결을 확인합니다. 아래 결과는 설명 예제 검증이며 연습문제 채점이 아닙니다. 핵심은 **무엇을 학습했고 무엇으로 선택했는지, 어떤 행렬을 재구성했는지**를 말할 수 있는 것입니다.


In [ ]:
assert len(review['history']) == 13
assert review['best_epoch'] == int(review['history'].validation_rmse.idxmin())
assert np.isclose(review['validation_metrics']['rmse'], review['history'].validation_rmse.min())
np.testing.assert_allclose(review['svd']['s'], [6., 2.])
np.testing.assert_allclose(review['svd']['reconstruction'], np.full((2,2), 3.))
pd.testing.assert_frame_equal(review['history'], full_rank['history'])
_callback = next(f.fn for f in app.fns.values() if f.api_name == 'review_compare')
_callback_output = _callback(2, .1, 2, 1)
assert len(_callback_output) == 5
assert my_report(mf_review_experiment(epochs=2, rank=1)['svd']) in _callback_output[0]
print('W08A_B_RUNTIME_CHECK_PASS: 선택 상태·SVD·앱 callback 확인')


## 참고자료와 다음 복습

- [이번 강의노트](https://github.com/lunalab-ai/recommender/blob/2026-fall-w08a/course/notion/sessions/w08a-mf-svd-review.md)
- [W05A 지표 보강 S1–S5](https://github.com/lunalab-ai/recommender/blob/2026-fall-w08a/course/notion/supplements/w05a-metrics-remediation.md)
- [W05B SGD 워크북 S1–S3](https://github.com/lunalab-ai/recommender/blob/2026-fall-w08a/course/notion/supplements/w05b-sgd-workbook.md)
- [W06A 검증 워크북 W1–W3](https://github.com/lunalab-ai/recommender/blob/2026-fall-w08a/course/notion/supplements/w06a-evaluation-workbook.md)
- [W06B SVD 계산 복습](https://github.com/lunalab-ai/recommender/blob/2026-fall-w08a/course/notion/supplements/w06b-calculation-practice.md)
- [NumPy SVD 공식 문서](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html)
- [scikit-learn 데이터 누수 설명](https://scikit-learn.org/stable/common_pitfalls.html)

필수 실습은 작은 합성 자료만 사용합니다. 실제 MovieLens 자동 준비·캐시 실험은 기존 W05A/W06A의 선택 확장 경로를 이용합니다. 서로 다른 표본의 점수를 같은 통제 실험처럼 비교하지 않습니다.


## 마지막 앱 실행 · Q58 기록

Colab에서는 아래 셀이 임시 공유 주소와 화면을 만듭니다. 주소는 실행 중인 런타임에 의존하므로 배포 링크는 고정 notebook을 사용합니다. Q58 활동은 **한 설정 변경 → 관찰 → my_report 수정 → 앱 재생성 → 다시 실행** 순서입니다. 로컬 자동 검증은 서버를 열지 않고 위 callback을 직접 실행했습니다.


In [ ]:
if IN_COLAB:
    app.launch(share=True, debug=False, prevent_thread_lock=True)
else:
    print('로컬 검증: 서버 시작 없이 실제 callback 검사를 마쳤습니다.')
    app.close()
